In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import polyase as poly

In [ ]:
DIR="LR_DESIREE_PAPER"

### Overlap significant genes

In [ ]:
## read in cis results from desiree and atlantic
cis_results_desiree = pd.read_csv(f"{DIR}/ANALYSIS/ASE_results/comparison_DES_ATL/desiree_pairwise_cis_results_significant.csv")
cis_results_atlantic = pd.read_csv(f"{DIR}/ANALYSIS/ASE_results/comparison_DES_ATL/atlantic_pairwise_cis_results_significant.csv")
#  get unique gene_id for both dataframes
cis_results_desiree['Unitato_gene_id'] = cis_results_desiree['gene_id_i'].apply(lambda x: x.rsplit('.', 1)[0] if isinstance(x, str) else x)
cis_results_atlantic['Unitato_gene_id'] = cis_results_atlantic['gene_id_i'].apply(lambda x: x.rsplit('.', 1)[0] if isinstance(x, str) else x)
# merge on gene_id, allele_i, allele_j
merged_df = pd.merge(cis_results_desiree, cis_results_atlantic, on=["Unitato_gene_id"], suffixes=("_desiree", "_atlantic"))

In [ ]:
# read in the promotor comparison data
promoter_comparison = pd.read_csv(f"{DIR}/DOWNSTREAM_ANALYSIS/shared_atlantic_desiree/BLAST_promotor_ATL_vs_DES_filtered_all.tsv", sep="\t", header=None, names=["Des_gene_id", "Atl_gene_id", "percent_identity", "alignment_length", "mismatches", "gap_opens", "q_start", "q_end", "s_start", "s_end", "evalue", "bit_score"])
# remove prefix Des_gene_id and Atl_gene_id
promoter_comparison["Des_gene_id"] = promoter_comparison["Des_gene_id"].str.replace("DES.", "")
promoter_comparison["Atl_gene_id"] = promoter_comparison["Atl_gene_id"].str.replace("ATL.", "")

# remove the .1. Des_gene_id and Atl_gene_id
promoter_comparison["Des_gene_id"] = promoter_comparison["Des_gene_id"].str.replace(".1.", ".")
promoter_comparison["Des_gene_id"] = promoter_comparison["Des_gene_id"].str.replace(".2.", ".")
promoter_comparison["Des_gene_id"] = promoter_comparison["Des_gene_id"].str.replace(".3.", ".")
promoter_comparison["Des_gene_id"] = promoter_comparison["Des_gene_id"].str.replace(".4.", ".")
promoter_comparison["Des_gene_id"] = promoter_comparison["Des_gene_id"].str.replace(".5.", ".")
promoter_comparison["Atl_gene_id"] = promoter_comparison["Atl_gene_id"].str.replace(".1.", ".")
promoter_comparison["Atl_gene_id"] = promoter_comparison["Atl_gene_id"].str.replace(".2.", ".")
promoter_comparison["Atl_gene_id"] = promoter_comparison["Atl_gene_id"].str.replace(".3.", ".")
promoter_comparison["Atl_gene_id"] = promoter_comparison["Atl_gene_id"].str.replace(".4.", ".")
promoter_comparison["Atl_gene_id"] = promoter_comparison["Atl_gene_id"].str.replace(".5.", ".")

# filter for percent identity = 100 and alignment length 1500
promoter_comparison_filtered = promoter_comparison[(promoter_comparison["percent_identity"] == 100  ) & (promoter_comparison["alignment_length"] == 1500)]
promoter_comparison_filtered = promoter_comparison_filtered[
    (promoter_comparison_filtered["Des_gene_id"].isin(merged_df["gene_id_i_desiree"]) |
     promoter_comparison_filtered["Des_gene_id"].isin(merged_df["gene_id_j_desiree"])) |
    (promoter_comparison_filtered["Atl_gene_id"].isin(merged_df["gene_id_i_atlantic"]) |
     promoter_comparison_filtered["Atl_gene_id"].isin(merged_df["gene_id_j_atlantic"]))
]
promoter_comparison_filtered.drop_duplicates(subset=["Des_gene_id"], inplace=True)
promoter_comparison_filtered.drop_duplicates(subset=["Atl_gene_id"], inplace=True)

In [ ]:
promoter_comparison_filtered["Unitato_gene_id"] = promoter_comparison_filtered["Des_gene_id"].apply(lambda x: x.rsplit('.', 1)[0] if isinstance(x, str) else x)

# keep only Unitato_gene_ids that appear exactly twice (two haplotype pairs)
promoter_comparison_filtered = promoter_comparison_filtered[
    promoter_comparison_filtered["Unitato_gene_id"].isin(
        promoter_comparison_filtered["Unitato_gene_id"].value_counts()[
            promoter_comparison_filtered["Unitato_gene_id"].value_counts() == 2
        ].index
    )
]

# build valid allele-pair sets from merged_df (both orientations)
valid_des_pairs = (
    set(zip(merged_df["gene_id_i_desiree"], merged_df["gene_id_j_desiree"])) |
    set(zip(merged_df["gene_id_j_desiree"], merged_df["gene_id_i_desiree"]))
)
valid_atl_pairs = (
    set(zip(merged_df["gene_id_i_atlantic"], merged_df["gene_id_j_atlantic"])) |
    set(zip(merged_df["gene_id_j_atlantic"], merged_df["gene_id_i_atlantic"]))
)

def has_valid_pair(group):
    des_ids = group["Des_gene_id"].tolist()
    atl_ids = group["Atl_gene_id"].tolist()
    return (
        len(des_ids) >= 2 and len(atl_ids) >= 2 and
        (des_ids[0], des_ids[1]) in valid_des_pairs and
        (atl_ids[0], atl_ids[1]) in valid_atl_pairs
    )

promoter_comparison_filtered = promoter_comparison_filtered.groupby("Unitato_gene_id", group_keys=False).filter(has_valid_pair)

genes = []
for gene_id, group in promoter_comparison_filtered.groupby("Unitato_gene_id"):
    atl_ids = group["Atl_gene_id"].tolist()
    des_ids = group["Des_gene_id"].tolist()
    genes.append({
        "name": gene_id,
        "ATL": atl_ids[:2] if len(atl_ids) >= 2 else [atl_ids[0], atl_ids[0]],
        "DES": des_ids[:2] if len(des_ids) >= 2 else [des_ids[0], des_ids[0]],
    })

ASE_genes = [gene for gene in genes if "08G" not in gene["name"]]

### Remove genes from Chr08

In [ ]:
# read in gene counts for desiree and atlantic
ATL_counts = pd.read_csv("nf-core-plantlongrnaseq/output_atlantic_liftoff_CDS2exon/tximeta/all_samples.gene_counts.tsv",  sep="\t")
# convert to CPM counts
ATL_counts[ATL_counts.columns[2:]] = ATL_counts[ATL_counts.columns[2:]].div(ATL_counts[ATL_counts.columns[2:]].sum()) * 1000000
DES_counts = pd.read_csv("nf-core-plantlongrnaseq/output_desiree_liftoff_phased_CDS2exon/tximeta/all_samples.gene_counts.tsv", sep = "\t")
# convert to CPM counts
DES_counts[DES_counts.columns[2:]] = DES_counts[DES_counts.columns[2:]].div(DES_counts[DES_counts.columns[2:]].sum()) * 1000000
# subset for the sameples with "C_H" in the name and the gene_id column for both dataframes 
ATL_counts = ATL_counts[['gene_id', 'SRR14993892', 'SRR14993893', 'SRR14993894','SRR14993895', "SRR14996168" ]]
ATL_counts = ATL_counts[['gene_id', 'SRR14993894','SRR14993895', "SRR14996168" ]]
DES_counts =DES_counts[['gene_id', 'DE_C_H_13', 'DE_C_H_14', 'DE_C_H_15', 'DE_C_H_16']]

In [ ]:
# filter for log2FC !-inf and > -inf
merged_df = merged_df.replace([np.inf, -np.inf], np.nan).dropna(subset=['log2FC_desiree', 'log2FC_atlantic'])

# extract significant genes in both datasets
merged_df["gene_id"] = merged_df["Unitato_gene_id"]

#pd.DataFrame(merged_df["gene_id"].unique()).to_csv(f"{DIR}/ANALYSIS/ASE_results/comparison_DES_ATL/AT_DES_merged_cis_results_control_grouped_significant_gene_ids.csv", index=False, header=False)

In [ ]:
# set plot size for paper figures
sns.set(rc={'figure.figsize':(3,3)}, style="white")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(genes), figsize=(2 * len(genes), 3.5))

for col, gene in enumerate(genes):
    ax = axes[col]
    frames = []
    all_haps = []
    number_labels = []
    cultivar_positions = []  # (cultivar, center_x)
    pos = 0
    for cultivar, hap_list, counts_df in [
        ("DES", gene["DES"], DES_counts),
        ("ATL", gene["ATL"], ATL_counts),
    ]:
        c1 = counts_df[counts_df['gene_id'] == hap_list[0]].iloc[:, 1:].values.flatten().astype(float)
        c2 = counts_df[counts_df['gene_id'] == hap_list[1]].iloc[:, 1:].values.flatten().astype(float)

        # unique key per cultivar so shared haplotype IDs don't merge
        keys = [f"{cultivar}::{h}" for h in hap_list]

        frames.append(pd.DataFrame({
            'haplotype': [keys[0]]*len(c1) + [keys[1]]*len(c2),
            'ratio': np.concatenate([c1, c2])
        }))
        all_haps.extend(keys)
        # just the number, drop "Hap"/text
        number_labels.extend([''.join(filter(str.isdigit, h.split('.')[-1])) for h in hap_list])
        # center of this cultivar's two boxes
        cultivar_positions.append((cultivar, pos + 0.5 ))
        pos += 2

    df = pd.concat(frames)

    sns.boxplot(data=df, x='haplotype', y='ratio', order=all_haps,
                palette=['#D55E00', 'white', '#D55E00', 'white'], linewidth=1.2, ax=ax)
    sns.stripplot(data=df, x='haplotype', y='ratio', order=all_haps, color='black', size=7, alpha=1, jitter=True, ax=ax)
    ax.set_xticklabels(number_labels, fontsize=14)
    ax.set_yticklabels(ax.get_yticks().astype(int), fontsize=14)
    ax.set_xlabel("")
    ax.set_ylabel("", fontsize=16)
    ax.set_title(gene["name"], fontsize=12)

    # cultivar label once per pair, below the number labels
    for cultivar, x_center in cultivar_positions:
        ax.text(x_center, -0.15, cultivar, transform=ax.get_xaxis_transform(),
                ha='center', va='top', fontsize=16)

plt.tight_layout()
plt.savefig(f"{DIR}/ANALYSIS/final_plots_svg/AT_DES_shared_ASE_promoter.svg")

# Overlap all shared promotors (also non significant)

In [ ]:
## read in cis results from desiree and atlantic
cis_results_desiree = pd.read_csv(f"{DIR}/ANALYSIS/ASE_results/comparison_DES_ATL/desiree_pairwise_cis_results_all.csv")
cis_results_atlantic = pd.read_csv(f"{DIR}/ANALYSIS/ASE_results/comparison_DES_ATL/atlantic_pairwise_cis_results_all.csv")
#  get unique gene_id for both dataframes
cis_results_desiree['Unitato_gene_id'] = cis_results_desiree['gene_id_i'].apply(lambda x: x.rsplit('.', 1)[0] if isinstance(x, str) else x)
cis_results_atlantic['Unitato_gene_id'] = cis_results_atlantic['gene_id_i'].apply(lambda x: x.rsplit('.', 1)[0] if isinstance(x, str) else x)
# merge on gene_id, allele_i, allele_j
merged_df = pd.merge(cis_results_desiree, cis_results_atlantic, on=["Unitato_gene_id"], suffixes=("_desiree", "_atlantic"))
# export merged dataframe
#merged_df.to_csv(f"{DIR}/ANALYSIS/ASE_results/comparison_DES_ATL/AT_DES_merged_cis_results_pairwise.csv", index=False)


In [ ]:
len(cis_results_desiree['Unitato_gene_id'].unique())

In [ ]:
len(cis_results_atlantic['Unitato_gene_id'].unique())

In [ ]:
# count the number of unique combinations of gene_id_i and gene_id_j in the merged dataframe
# number of comparisons in both
print("Number of unique comparisons:", merged_df[["gene_id_i_desiree", "gene_id_j_desiree"]].drop_duplicates().shape[0])

In [ ]:
print("Number of genes:", merged_df[["Unitato_gene_id"]].drop_duplicates().shape[0])

In [ ]:
# filter for log2FC !-inf and > -inf
merged_df = merged_df.replace([np.inf, -np.inf], np.nan).dropna(subset=['log2FC_desiree', 'log2FC_atlantic'])

# extract significant genes in both datasets
merged_df["gene_id"] = merged_df["Unitato_gene_id"]

pd.DataFrame(merged_df["gene_id"].unique()).to_csv(f"{DIR}/ANALYSIS/ASE_results/comparison_DES_ATL/AT_DES_merged_cis_results_control_grouped_all_gene_ids.csv", index=False, header=False)

In [ ]:
merged_df["gene_id"].nunique()

In [ ]:
# read in the promotor comparison data
promoter_comparison = pd.read_csv(f"{DIR}/DOWNSTREAM_ANALYSIS/shared_atlantic_desiree/BLAST_promotor_ATL_vs_DES_filtered_all.tsv", sep="\t", header=None, names=["Des_gene_id", "Atl_gene_id", "percent_identity", "alignment_length", "mismatches", "gap_opens", "q_start", "q_end", "s_start", "s_end", "evalue", "bit_score"])
# remove prefix Des_gene_id and Atl_gene_id
promoter_comparison["Des_gene_id"] = promoter_comparison["Des_gene_id"].str.replace("DES.", "")
promoter_comparison["Atl_gene_id"] = promoter_comparison["Atl_gene_id"].str.replace("ATL.", "")

# remove the .1. Des_gene_id and Atl_gene_id
promoter_comparison["Des_gene_id"] = promoter_comparison["Des_gene_id"].str.replace(".1.", ".")
promoter_comparison["Des_gene_id"] = promoter_comparison["Des_gene_id"].str.replace(".2.", ".")
promoter_comparison["Des_gene_id"] = promoter_comparison["Des_gene_id"].str.replace(".3.", ".")
promoter_comparison["Des_gene_id"] = promoter_comparison["Des_gene_id"].str.replace(".4.", ".")
promoter_comparison["Atl_gene_id"] = promoter_comparison["Atl_gene_id"].str.replace(".1.", ".")
promoter_comparison["Atl_gene_id"] = promoter_comparison["Atl_gene_id"].str.replace(".2.", ".")
promoter_comparison["Atl_gene_id"] = promoter_comparison["Atl_gene_id"].str.replace(".3.", ".")
promoter_comparison["Atl_gene_id"] = promoter_comparison["Atl_gene_id"].str.replace(".4.", ".")

# filter for percent identity = 100 and alignment length 1500
promoter_comparison_filtered = promoter_comparison[(promoter_comparison["mismatches"] == 0) & (promoter_comparison["alignment_length"] == 1500)]
promoter_comparison_filtered = promoter_comparison_filtered[
    (promoter_comparison_filtered["Des_gene_id"].isin(merged_df["gene_id_i_desiree"]) |
     promoter_comparison_filtered["Des_gene_id"].isin(merged_df["gene_id_j_desiree"])) &
    (promoter_comparison_filtered["Atl_gene_id"].isin(merged_df["gene_id_i_atlantic"]) |
     promoter_comparison_filtered["Atl_gene_id"].isin(merged_df["gene_id_j_atlantic"]))
]
promoter_comparison_filtered.drop_duplicates(subset=["Des_gene_id"], inplace=True)
promoter_comparison_filtered.drop_duplicates(subset=["Atl_gene_id"], inplace=True)


In [ ]:
promoter_comparison_filtered["Unitato_gene_id"] = promoter_comparison_filtered["Des_gene_id"].apply(lambda x: x.rsplit('.', 1)[0] if isinstance(x, str) else x)

# keep only Unitato_gene_ids that appear exactly twice (two haplotype pairs)
promoter_comparison_filtered = promoter_comparison_filtered[
    promoter_comparison_filtered["Unitato_gene_id"].isin(
        promoter_comparison_filtered["Unitato_gene_id"].value_counts()[
            promoter_comparison_filtered["Unitato_gene_id"].value_counts() == 2
        ].index
    )
]

# build valid allele-pair sets from merged_df (both orientations)
valid_des_pairs = (
    set(zip(merged_df["gene_id_i_desiree"], merged_df["gene_id_j_desiree"])) |
    set(zip(merged_df["gene_id_j_desiree"], merged_df["gene_id_i_desiree"]))
)
valid_atl_pairs = (
    set(zip(merged_df["gene_id_i_atlantic"], merged_df["gene_id_j_atlantic"])) |
    set(zip(merged_df["gene_id_j_atlantic"], merged_df["gene_id_i_atlantic"]))
)

def has_valid_pair(group):
    des_ids = group["Des_gene_id"].tolist()
    atl_ids = group["Atl_gene_id"].tolist()
    return (
        len(des_ids) >= 2 and len(atl_ids) >= 2 and
        (des_ids[0], des_ids[1]) in valid_des_pairs and
        (atl_ids[0], atl_ids[1]) in valid_atl_pairs
    )

promoter_comparison_filtered = promoter_comparison_filtered.groupby("Unitato_gene_id", group_keys=False).filter(has_valid_pair)

genes = []
for gene_id, group in promoter_comparison_filtered.groupby("Unitato_gene_id"):
    atl_ids = group["Atl_gene_id"].tolist()
    des_ids = group["Des_gene_id"].tolist()
    genes.append({
        "name": gene_id,
        "ATL": atl_ids[:2] if len(atl_ids) >= 2 else [atl_ids[0], atl_ids[0]],
        "DES": des_ids[:2] if len(des_ids) >= 2 else [des_ids[0], des_ids[0]],
    })

genes = [gene for gene in genes if "08G" not in gene["name"]]

In [ ]:
promoter_comparison_filtered.sort_values(by="percent_identity", ascending=False, inplace=True)
# save promoter_comparison_filtered to csv
promoter_comparison_filtered.to_csv(f"{DIR}/ANALYSIS/final_tables_paper/promoter_comparison_filtered_all.csv", index=False)

In [ ]:
merged_df[(merged_df['gene_id_i_desiree'].isin(promoter_comparison_filtered['Des_gene_id'].values)) | (merged_df['gene_id_j_desiree'].isin(promoter_comparison_filtered['Des_gene_id'].values) & (merged_df['gene_id_i_atlantic'].isin(promoter_comparison_filtered['Atl_gene_id'].values)) | (merged_df['gene_id_j_atlantic'].isin(promoter_comparison_filtered['Atl_gene_id'].values)))].drop_duplicates(inplace=False)

In [ ]:

# extract significant genes in both datasets
merged_df["gene_id"] = merged_df["Unitato_gene_id"]

In [ ]:
from scipy import stats

#sample_cols_atl = ['SRR14993892', 'SRR14993893', 'SRR14993894', 'SRR14993895', 'SRR14996168']
sample_cols_atl = [ 'SRR14993894', 'SRR14993895', 'SRR14996168']
sample_cols_des = ['DE_C_H_13', 'DE_C_H_14', 'DE_C_H_15', 'DE_C_H_16']

atl_idx = ATL_counts.set_index('gene_id')[sample_cols_atl]
des_idx = DES_counts.set_index('gene_id')[sample_cols_des]

def count_logfc(counts_idx, id_i, id_j, pseudocount=0):
    try:
        c_i = counts_idx.loc[id_i].values.astype(float)
        c_j = counts_idx.loc[id_j].values.astype(float)
    except KeyError:
        return np.nan
    return np.log2((np.mean(c_i) + pseudocount) / (np.mean(c_j) + pseudocount))

# build allele-order lookup from the significant genes dict
genes_order = {g["name"]: g for g in genes}

rows = []
for _, grp in merged_df.groupby("Unitato_gene_id"):
    uid = grp["Unitato_gene_id"].iloc[0]
    if uid in genes_order:
        g = genes_order[uid]
        atl_fc = count_logfc(atl_idx, g["ATL"][0], g["ATL"][1])
        des_fc = count_logfc(des_idx, g["DES"][0], g["DES"][1])
        rows.append({"Unitato_gene_id": uid, "log2FC_atl": atl_fc, "log2FC_des": des_fc})

fc_df = pd.DataFrame(rows).dropna(subset=["log2FC_atl", "log2FC_des"])

fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(
    fc_df["log2FC_des"],
    fc_df["log2FC_atl"],
    alpha=0.3, s=50, c = "black", 
)

lim = max(abs(fc_df["log2FC_des"]).max(), abs(fc_df["log2FC_atl"]).max()) * 1.1
ax.plot([-lim, lim], [-lim, lim], "k--", lw=0.8, alpha=0.2)

ax.plot([-lim, lim], [-1, -1], "k--", lw=0.8, alpha=0.2)
ax.plot([-lim, lim], [1, 1], "k--", lw=0.8, alpha=0.2)


ax.plot([-1, -1], [-lim, lim], "k--", lw=0.8, alpha=0.2)
ax.plot([1, 1], [-lim, lim], "k--", lw=0.8, alpha=0.2)
ax.set_xlim(-3.3, 7)
ax.set_ylim(-3.3, 7)
# set ticks at -3, -2, -1, 0, 1, 2, 3
ax.set_xticks([ -2, 0, 2,4,6 ])
ax.set_yticks([ -2,  0, 2,4,6 ])
ax.set_xlabel("Désirée log$_2$FC(allele i/allele j)")
ax.set_ylabel("Atlantic  log$_2$FC(allele i/allele j)")
# set x and y ticks to fontsize 12
ax.tick_params(axis='both', which='major', labelsize=13)
#ax.set_title("Shared alleles — log2FC", fontsize=11)
# to the pearson correlation coefficient and p-value
r, p = stats.pearsonr(fc_df["log2FC_des"], fc_df["log2FC_atl"])
ax.text(0.05, 0.95, f"r = {r:.2f}, p = {p:.3}", transform=ax.transAxes, fontsize = 12, va="top")

# color the genes that are in the genes list
for _, row in fc_df.iterrows():
    if row["Unitato_gene_id"] in [gene["name"] for gene in ASE_genes]:
        ax.scatter(row["log2FC_des"], row["log2FC_atl"], color="#D55E00", s=50,  zorder=4, alpha=0.5)
        # also label the gene name
        #ax.text(row["log2FC_des"] + 0.1, row["log2FC_atl"] + 0.1, row["Unitato_gene_id"], fontsize=8, color="black")  
sns.despine()
plt.tight_layout()

# save as svg
plt.savefig(f"{DIR}/ANALYSIS/final_plots_svg/Promotor_comparison_ATL_DES.svg", format="svg")
plt.show()

In [ ]:
rows = []
for _, grp in merged_df.groupby("Unitato_gene_id"):
    uid = grp["Unitato_gene_id"].iloc[0]
    if uid in genes_order:
        g = genes_order[uid]
        atl_fc = count_logfc(atl_idx, g["ATL"][0], g["ATL"][1])
        des_fc = count_logfc(des_idx, g["DES"][0], g["DES"][1])
        rows.append({
            "Unitato_gene_id": uid,
            "log2FC_atl": atl_fc,
            "log2FC_des": des_fc,
            "ATL_allele_i": g["ATL"][0],
            "ATL_allele_j": g["ATL"][1],
            "DES_allele_i": g["DES"][0],
            "DES_allele_j": g["DES"][1],
        })

fc_df = pd.DataFrame(rows).dropna(subset=["log2FC_atl", "log2FC_des"])

In [ ]:
fc_df.to_csv(f"{DIR}/ANALYSIS/final_tables_paper/Promotor_comparison_ATL_DES_log2FC.csv", index=False)